# routes.router

> Convenience router factory that wires up standard virtual collection routes (Tier 2 API).

In [ ]:
#| default_exp routes.router

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Callable, List, Tuple

from fasthtml.common import APIRouter

from cjm_fasthtml_virtual_collection.core.models import (
    VirtualCollectionConfig, VirtualCollectionState, VirtualCollectionUrls,
)
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds
from cjm_fasthtml_virtual_collection.routes.handlers import (
    handle_navigate, handle_navigate_to_index, handle_update_viewport,
    handle_focus_row,
)

In [ ]:
#| export
def init_virtual_collection_router(
    config: VirtualCollectionConfig,                    # Collection config
    state_getter: Callable[[], VirtualCollectionState],  # Function to get current state
    state_setter: Callable[[VirtualCollectionState], None],  # Function to save state
    get_items: Callable[[], list],                       # Function to get current items
    render_cell: Callable,                               # Cell render callback
    route_prefix: str = "/collection",                   # Route prefix
) -> Tuple[APIRouter, VirtualCollectionUrls]:  # (router, urls) tuple
    """Initialize an APIRouter with all standard virtual collection routes."""
    router = APIRouter(prefix=route_prefix)
    ids = VirtualCollectionHtmlIds(prefix=config.prefix)

    # focus_row URL is resolved after route registration (forward reference)
    _focus_url = ""

    def _get_focus_url() -> str:
        return _focus_url

    # -----------------------------------------------------------------
    # Navigation routes
    # -----------------------------------------------------------------

    def _nav(direction: str) -> Any:
        """Shared navigation handler."""
        state = state_getter()
        items = get_items()
        result = handle_navigate(
            direction=direction, items=items, state=state,
            config=config, ids=ids, render_cell=render_cell,
            focus_url=_get_focus_url(),
        )
        state_setter(state)
        return result

    @router
    def nav_up() -> Any: return _nav("up")

    @router
    def nav_down() -> Any: return _nav("down")

    @router
    def nav_first() -> Any: return _nav("first")

    @router
    def nav_last() -> Any: return _nav("last")

    @router
    def nav_page_up() -> Any: return _nav("page_up")

    @router
    def nav_page_down() -> Any: return _nav("page_down")

    @router
    def nav_to_index(target_index: int) -> Any:
        """Navigate to a specific window_start index."""
        state = state_getter()
        items = get_items()
        result = handle_navigate_to_index(
            target_index=target_index, items=items, state=state,
            config=config, ids=ids, render_cell=render_cell,
            focus_url=_get_focus_url(),
        )
        state_setter(state)
        return result

    # -----------------------------------------------------------------
    # Viewport route
    # -----------------------------------------------------------------

    @router
    def update_viewport(visible_rows: int, is_auto: str = "true") -> Any:
        """Update viewport with new row count."""
        state = state_getter()
        items = get_items()
        result = handle_update_viewport(
            visible_rows=visible_rows, items=items, state=state,
            config=config, ids=ids, render_cell=render_cell,
            is_auto=(is_auto == "true"),
            focus_url=_get_focus_url(),
        )
        state_setter(state)
        return result

    # -----------------------------------------------------------------
    # Focus route
    # -----------------------------------------------------------------

    @router
    def focus_row(row_index: int) -> Any:
        """Move cursor to a specific row via click/tap."""
        state = state_getter()
        items = get_items()
        result = handle_focus_row(
            row_index=row_index, items=items, state=state,
            config=config, ids=ids, render_cell=render_cell,
            focus_url=_get_focus_url(),
        )
        state_setter(state)
        return result

    # -----------------------------------------------------------------
    # Build URL bundle from registered routes
    # -----------------------------------------------------------------

    _focus_url = focus_row.to()

    urls = VirtualCollectionUrls(
        nav_up=nav_up.to(),
        nav_down=nav_down.to(),
        nav_page_up=nav_page_up.to(),
        nav_page_down=nav_page_down.to(),
        nav_first=nav_first.to(),
        nav_last=nav_last.to(),
        nav_to_index=nav_to_index.to(),
        update_viewport=update_viewport.to(),
        focus_row=_focus_url,
    )

    return router, urls

## Tests

In [ ]:
from dataclasses import dataclass
from fasthtml.common import Span
from cjm_fasthtml_virtual_collection.core.models import (
    VirtualCollectionConfig, VirtualCollectionState, ColumnDef,
)

@dataclass
class Item:
    name: str

test_items = [Item(f"file_{i}.txt") for i in range(50)]
columns = (ColumnDef(key="name", header="Name", width="1fr"),)
config = VirtualCollectionConfig(prefix="tr", columns=columns)
state = VirtualCollectionState(total_items=50, visible_rows=5)

def test_render_cell(item, ctx):
    return Span(item.name)

router, urls = init_virtual_collection_router(
    config=config,
    state_getter=lambda: state,
    state_setter=lambda s: None,
    get_items=lambda: test_items,
    render_cell=test_render_cell,
    route_prefix="/vc",
)

# Verify URL bundle is populated
assert urls.nav_up != ""
assert urls.nav_down != ""
assert urls.nav_page_up != ""
assert urls.nav_page_down != ""
assert urls.nav_first != ""
assert urls.nav_last != ""
assert urls.nav_to_index != ""
assert urls.update_viewport != ""

print(f"Router prefix: /vc")
print(f"URLs:")
for field in ['nav_up', 'nav_down', 'nav_page_up', 'nav_page_down', 'nav_first', 'nav_last', 'nav_to_index', 'update_viewport']:
    print(f"  {field}: {getattr(urls, field)}")

Router prefix: /vc
URLs:
  nav_up: /vc/nav_up
  nav_down: /vc/nav_down
  nav_page_up: /vc/nav_page_up
  nav_page_down: /vc/nav_page_down
  nav_first: /vc/nav_first
  nav_last: /vc/nav_last
  nav_to_index: /vc/nav_to_index
  update_viewport: /vc/update_viewport


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()